# Final Model Synthesis

This notebook compares the supervised, boosting, and semi-supervised approaches evaluated in the project.

The goal is to identify the strongest candidate models using the common cross-validation results, then evaluate the selected final model(s) on the untouched test set.

Model selection is based only on the training-set cross-validation results. The test set is used only once at the end for final unbiased evaluation.

In [3]:
import pandas as pd
import numpy as np

from helpers.utils import *

## 1. Load Cross-Validation Results

The project notebooks save their final cross-validation results in a shared `results.csv` file. These results are used to compare the different model families before touching the final test set.

In [4]:
results = pd.read_csv("results.csv")

print("Number of rows:", len(results))
print("\nModels found:")
for model in results["model"].unique():
    print("-", model)

Number of rows: 340

Models found:
- AdaBoost Tuned
- Bagging
- Baseline (mean)
- Bayesian Ridge
- Extra-Trees
- Gradient Boosting Tuned
- HistGradientBoosting Tuned
- Linear regression
- Linear regression (backward)
- Linear regression (forward)
- Naive Bayes classifier
- Naive bayesian
- PCR
- Pruned CART
- Pruned tree classifier
- Random forest
- Random forest classifier
- SVM
- SVM classifier
- k-NN classifier
- kNN regression


In [5]:
print("\nNumber of models:", results["model"].nunique())

results[
    results["family"] == "all"
][
    [
        "model",
        "target",
        "MAE",
        "RMSE",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good"
    ]
]


Number of models: 21


,model,target,MAE,RMSE,F1_nc,recall_nc,accuracy,precision_good
0,AdaBoost Tuned,yield_strength_MPa,43.7947,57.8942,0.0194,0.0098,0.8368,0.8366
4,AdaBoost Tuned,uts_MPa,40.2143,51.4968,0.5793,0.4242,0.7950,0.7734
8,AdaBoost Tuned,elongation_pct,2.4142,3.0266,0.1818,0.1042,0.9192,0.9218
12,AdaBoost Tuned,charpy_toughness_J,27.3719,35.7354,0.0000,0.0000,0.9079,0.9079
16,AdaBoost Tuned,label,NaN,NaN,0.3061,0.1807,0.6114,0.5750
...,...,...,...,...,...,...,...,...
320,kNN regression,yield_strength_MPa,39.4815,58.4946,0.7000,0.6863,0.9031,0.9386
324,kNN regression,uts_MPa,34.0177,52.0473,0.6578,0.6263,0.7832,0.8221
328,kNN regression,elongation_pct,2.1612,2.9260,0.5823,0.4792,0.9408,0.9525
332,kNN regression,charpy_toughness_J,18.0415,34.1080,0.7812,0.8621,0.9556,0.9857


## 2. Global Cross-Validation Comparison

The models are first compared using the common cross-validation results on the training set.

Regression models are compared separately for each mechanical property using MAE and RMSE. The final weld conformity decision is then compared across both regression-based approaches and direct classifiers using F1-score, recall, accuracy, and precision.

In [6]:
# Overall results only — exclude family-specific rows
overall = results[results["family"] == "all"].copy()

print("Number of overall results:", len(overall))
print("Number of models:", overall["model"].nunique())

Number of overall results: 85
Number of models: 21


In [7]:
regression_targets = [
    "yield_strength_MPa",
    "uts_MPa",
    "elongation_pct",
    "charpy_toughness_J"
]

regression_comparison = overall[
    overall["target"].isin(regression_targets)
][
    ["model", "target", "MAE", "RMSE", "MAE_std", "RMSE_std"]
].copy()

regression_comparison

,model,target,MAE,RMSE,MAE_std,RMSE_std
0,AdaBoost Tuned,yield_strength_MPa,43.7947,57.8942,3.2775,6.4912
4,AdaBoost Tuned,uts_MPa,40.2143,51.4968,3.0389,6.8814
8,AdaBoost Tuned,elongation_pct,2.4142,3.0266,0.1302,0.1265
12,AdaBoost Tuned,charpy_toughness_J,27.3719,35.7354,3.2891,4.5335
20,Bagging,yield_strength_MPa,35.2531,49.4110,3.8001,6.8403
...,...,...,...,...,...,...
304,SVM,charpy_toughness_J,22.0207,31.3150,2.7431,4.2532
320,kNN regression,yield_strength_MPa,39.4815,58.4946,2.1721,8.5917
324,kNN regression,uts_MPa,34.0177,52.0473,2.2443,11.0708
328,kNN regression,elongation_pct,2.1612,2.9260,0.1580,0.3424


In [8]:
classification_comparison = overall[
    overall["target"] == "label"
][
    [
        "model",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good"
    ]
].copy()

classification_comparison = classification_comparison.sort_values(
    "F1_nc",
    ascending=False
)

classification_comparison

,model,F1_nc,recall_nc,accuracy,precision_good
116,Gradient Boosting Tuned,0.8590,0.8072,0.8743,0.8431
312,SVM classifier,0.8415,0.8313,0.8514,0.8511
96,Extra-Trees,0.8392,0.7229,0.8686,0.8000
136,HistGradientBoosting Tuned,0.8302,0.7952,0.8457,0.8283
36,Bagging,0.8108,0.7229,0.8400,0.7909
316,k-NN classifier,0.8000,0.8193,0.8057,0.8295
336,kNN regression,0.7805,0.7711,0.7943,0.7979
284,Random forest,0.7571,0.6386,0.8057,0.7458
288,Random forest classifier,0.7500,0.6867,0.7829,0.7547
260,Pruned CART,0.7458,0.7952,0.7429,0.7901


In [9]:
for target in regression_targets:
    print(f"\n### {target}")
    
    display(
        regression_comparison[
            regression_comparison["target"] == target
        ]
        .sort_values("MAE")
        .head(10)
    )


### yield_strength_MPa


,model,target,MAE,RMSE,MAE_std,RMSE_std
100,Gradient Boosting Tuned,yield_strength_MPa,28.6248,41.5338,2.2545,6.4211
80,Extra-Trees,yield_strength_MPa,29.0767,42.4853,2.7953,7.6437
120,HistGradientBoosting Tuned,yield_strength_MPa,30.8290,45.1285,3.9886,8.0586
292,SVM,yield_strength_MPa,32.9972,51.6460,4.6229,10.4611
268,Random forest,yield_strength_MPa,33.9149,47.4283,3.5882,6.7493
20,Bagging,yield_strength_MPa,35.2531,49.4110,3.8001,6.8403
320,kNN regression,yield_strength_MPa,39.4815,58.4946,2.1721,8.5917
0,AdaBoost Tuned,yield_strength_MPa,43.7947,57.8942,3.2775,6.4912
244,Pruned CART,yield_strength_MPa,48.6426,67.2701,6.3172,10.1378
180,Linear regression (forward),yield_strength_MPa,50.2624,67.3442,5.8832,5.0035



### uts_MPa


,model,target,MAE,RMSE,MAE_std,RMSE_std
84,Extra-Trees,uts_MPa,25.2362,38.8386,3.2438,9.4624
104,Gradient Boosting Tuned,uts_MPa,25.2797,40.8284,4.2293,10.7642
124,HistGradientBoosting Tuned,uts_MPa,26.5784,40.1272,3.1742,8.6362
296,SVM,uts_MPa,27.4000,45.3353,4.1871,11.1795
272,Random forest,uts_MPa,29.5455,42.2921,3.0259,8.4027
24,Bagging,uts_MPa,30.8448,44.4459,3.5125,8.4589
324,kNN regression,uts_MPa,34.0177,52.0473,2.2443,11.0708
4,AdaBoost Tuned,uts_MPa,40.2143,51.4968,3.0389,6.8814
248,Pruned CART,uts_MPa,41.7683,59.4561,5.9564,10.3356
164,Linear regression (backward),uts_MPa,42.2646,58.8834,3.4638,4.1230



### elongation_pct


,model,target,MAE,RMSE,MAE_std,RMSE_std
88,Extra-Trees,elongation_pct,1.7850,2.3563,0.1746,0.2593
300,SVM,elongation_pct,1.8319,2.5700,0.2066,0.5171
128,HistGradientBoosting Tuned,elongation_pct,1.9277,2.5122,0.1474,0.2084
276,Random forest,elongation_pct,1.9430,2.5071,0.1541,0.2236
28,Bagging,elongation_pct,1.9443,2.5143,0.1402,0.2221
108,Gradient Boosting Tuned,elongation_pct,1.9512,2.5538,0.2085,0.2453
328,kNN regression,elongation_pct,2.1612,2.9260,0.1580,0.3424
8,AdaBoost Tuned,elongation_pct,2.4142,3.0266,0.1302,0.1265
252,Pruned CART,elongation_pct,2.5676,3.2878,0.0924,0.1490
168,Linear regression (backward),elongation_pct,2.6881,3.4654,0.1851,0.2982



### charpy_toughness_J


,model,target,MAE,RMSE,MAE_std,RMSE_std
92,Extra-Trees,charpy_toughness_J,17.3670,27.7069,2.8468,5.0882
332,kNN regression,charpy_toughness_J,18.0415,34.1080,2.6982,5.2079
132,HistGradientBoosting Tuned,charpy_toughness_J,18.4703,27.5908,2.8453,5.4091
256,Pruned CART,charpy_toughness_J,18.8707,32.4594,3.5619,5.8135
280,Random forest,charpy_toughness_J,19.0877,28.3864,2.7894,4.6300
32,Bagging,charpy_toughness_J,19.1077,28.1598,2.8476,4.3602
112,Gradient Boosting Tuned,charpy_toughness_J,19.6093,28.7862,3.3133,5.3998
304,SVM,charpy_toughness_J,22.0207,31.3150,2.7431,4.2532
192,Linear regression (forward),charpy_toughness_J,25.1361,32.1089,2.5584,3.2838
172,Linear regression (backward),charpy_toughness_J,25.2942,31.9905,3.1391,3.6993


### 2.1 Regression Comparison Summary

The regression comparison shows that Extra-Trees provides the strongest overall predictive performance across the four mechanical properties.

- Gradient Boosting achieves the lowest MAE for yield strength.
- Extra-Trees achieves the lowest MAE for UTS, elongation, and Charpy toughness.
- Histogram Gradient Boosting remains competitive, especially for Charpy toughness.
- Linear models and AdaBoost perform substantially worse on most targets.

Based on cross-validation regression performance, Extra-Trees is the strongest overall regression candidate, while Gradient Boosting remains competitive for yield strength.

## 3. Family-Specific Performance

The strongest overall models are now examined separately for each alloy family. This helps identify whether a model performs consistently across C-Mn, CrMo2, and CrMo91 steels, or whether its strong global performance is mainly driven by the dominant C-Mn family.

In [10]:
top_models = [
    "Extra-Trees",
    "Gradient Boosting Tuned",
    "HistGradientBoosting Tuned",
    "SVM classifier"
]

family_comparison = results[
    results["model"].isin(top_models)
].copy()

family_comparison[
    [
        "model",
        "target",
        "family",
        "MAE",
        "RMSE",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good"
    ]
]

,model,target,family,MAE,RMSE,F1_nc,recall_nc,accuracy,precision_good
80,Extra-Trees,yield_strength_MPa,all,29.0767,42.4853,0.7811,0.6471,0.9402,0.9348
81,Extra-Trees,yield_strength_MPa,C-Mn,27.0808,36.8705,0.7811,0.6471,0.9330,0.9258
82,Extra-Trees,yield_strength_MPa,CrMo2,44.7174,75.6869,0.0000,0.0000,1.0000,1.0000
83,Extra-Trees,yield_strength_MPa,CrMo91,45.7218,71.9717,0.0000,0.0000,1.0000,1.0000
84,Extra-Trees,uts_MPa,all,25.2362,38.8386,0.7378,0.6111,0.8555,0.8344
...,...,...,...,...,...,...,...,...,...
139,HistGradientBoosting Tuned,label,CrMo91,NaN,NaN,0.0000,0.0000,0.3333,0.4000
312,SVM classifier,label,all,NaN,NaN,0.8415,0.8313,0.8514,0.8511
313,SVM classifier,label,C-Mn,NaN,NaN,0.8516,0.8354,0.8571,0.8471
314,SVM classifier,label,CrMo2,NaN,NaN,0.6667,1.0000,0.8750,1.0000


In [11]:
label_family_comparison = family_comparison[
    family_comparison["target"] == "label"
][
    [
        "model",
        "family",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good",
        "n_clf",
        "n_nc"
    ]
].sort_values(
    ["family", "F1_nc"],
    ascending=[True, False]
)

label_family_comparison

,model,family,F1_nc,recall_nc,accuracy,precision_good,n_clf,n_nc
117,Gradient Boosting Tuned,C-Mn,0.8649,0.8101,0.8758,0.8370,161,79
137,HistGradientBoosting Tuned,C-Mn,0.8571,0.8354,0.8634,0.8488,161,79
313,SVM classifier,C-Mn,0.8516,0.8354,0.8571,0.8471,161,79
97,Extra-Trees,C-Mn,0.8467,0.7342,0.8696,0.7961,161,79
118,Gradient Boosting Tuned,CrMo2,1.0000,1.0000,1.0000,1.0000,8,1
314,SVM classifier,CrMo2,0.6667,1.0000,0.8750,1.0000,8,1
98,Extra-Trees,CrMo2,0.0000,0.0000,0.8750,0.8750,8,1
138,HistGradientBoosting Tuned,CrMo2,0.0000,0.0000,0.8750,0.8750,8,1
99,Extra-Trees,CrMo91,0.8000,0.6667,0.8333,0.7500,6,3
119,Gradient Boosting Tuned,CrMo91,0.6667,0.6667,0.6667,0.6667,6,3


### 3.1 Family-Specific Comparison Summary

The strongest models show similar performance on the dominant C-Mn family. Gradient Boosting achieves the highest F1-score on C-Mn, while Histogram Gradient Boosting and the direct SVM classifier achieve slightly higher recall.

Results for CrMo2 and CrMo91 must be interpreted with caution because the number of evaluable welds is very small. For CrMo2, only 8 weld groups are evaluable and only 1 is non-conforming. For CrMo91, only 6 weld groups are evaluable and 3 are non-conforming. Consequently, a single prediction can strongly affect F1-score and recall.

The family analysis therefore does not change the global shortlist. Gradient Boosting, Extra-Trees, Histogram Gradient Boosting, and the direct SVM classifier remain the strongest candidates.

## 4. Semi-Supervised Methods

Two semi-supervised approaches were evaluated: clustering with K-means and self-training with pseudo-labels.

### K-means

The clusters contain some information about weld conformity, as cluster purity is consistently higher than with randomly shuffled labels. However, including the unlabelled rows does not improve classification performance. In most configurations, the F1-score decreases when unlabelled observations are included.

The labelled and unlabelled observations also appear to come from different regions of the feature space. A Random Forest can distinguish labelled from unlabelled rows with an AUC of 0.946, suggesting an important distribution shift between the two groups.

### Self-training

Self-training with pseudo-labels does not provide a meaningful improvement over the supervised reference model. Yield-strength and elongation errors slightly increase, while the improvement for UTS is very small and remains within the variability observed across folds.

For the final conformity label, self-training decreases the F1-score from 0.797 to 0.788 and recall from 0.663 to 0.651.

### Conclusion

The semi-supervised methods tested here do not improve the supervised models. Therefore, the final model selection is based on the supervised approaches.

## 5. Final Candidate Selection

Based on the cross-validation results, three models are retained for final evaluation on the untouched test set:

- **Extra-Trees** as the strongest overall regression model, with the lowest MAE for UTS, elongation, and Charpy toughness.
- **Gradient Boosting Tuned** as a strong alternative regression model, with the lowest MAE for yield strength and the best overall conformity-classification F1-score among the regression-based approaches.
- **SVM classifier** as the strongest direct-classification candidate, with the highest recall for non-conforming welds among the top-performing models.

The final test set is used only after this selection step.

In [12]:
final_candidates = pd.DataFrame({
    "role": [
        "Regression candidate",
        "Regression candidate",
        "Direct classification candidate"
    ],
    "model": [
        "Extra-Trees",
        "Gradient Boosting Tuned",
        "SVM classifier"
    ],
    "reason": [
        "Best MAE on 3 of 4 regression targets",
        "Best yield MAE and best overall label F1 among regression approaches",
        "Highest recall among top final-label models"
    ]
})

final_candidates

,role,model,reason
0,Regression candidate,Extra-Trees,Best MAE on 3 of 4 regression targets
1,Regression candidate,Gradient Boosting Tuned,Best yield MAE and best overall label F1 among...
2,Direct classification candidate,SVM classifier,Highest recall among top final-label models


## 6. Final Evaluation on the Untouched Test Set

The final candidate models were selected using cross-validation on the training set only.

The untouched test set is now used for the first time to obtain an unbiased estimate of final model performance.

In [13]:
from pathlib import Path

from helpers.utils import (
    load_train,
    add_family,
    TARGETS,
    input_cols,
    build_pipeline,
    true_criteria,
    predicted_criteria,
    deposit_labels,
    deposit_data,
    DEPOSIT_COLS
)

train = load_train()

test_path = Path("../data/test.csv")
test = pd.read_csv(test_path)

test["family"] = add_family(test)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("\nTest families:")
print(test["family"].value_counts())

Train rows: 1332
Test rows: 307

Test families:
family
C-Mn      233
CrMo2      52
CrMo91     21
CrMo9       1
Name: count, dtype: int64


In [14]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GridSearchCV

extra_trees_searches = {}
extra_trees_best = {}

param_grid_extra = {
    "model__max_features": [0.1, 0.2, 0.33, 0.5, 0.75, 1.0],
    "model__min_samples_leaf": [1, 2, 4, 8]
}

for target in TARGETS:
    X, y, cv = get_xy(train, target)

    search = GridSearchCV(
        build_pipeline(
            ExtraTreesRegressor(
                n_estimators=200,
                random_state=42
            )
        ),
        param_grid_extra,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    search.fit(X, y)

    extra_trees_searches[target] = search
    extra_trees_best[target] = search.best_estimator_

    print(f"\n{target}")
    print("Best parameters:", search.best_params_)
    print("CV MAE:", -search.best_score_)


yield_strength_MPa
Best parameters: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}
CV MAE: 29.049929646599093

uts_MPa
Best parameters: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}
CV MAE: 25.190310026510808

elongation_pct
Best parameters: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}
CV MAE: 1.7838207609806471

charpy_toughness_J
Best parameters: {'model__max_features': 1.0, 'model__min_samples_leaf': 1}
CV MAE: 17.415859564792353


### 6.1 Extra-Trees Test Evaluation

The best Extra-Trees configuration for each regression target is retrained on all available training observations for that target and evaluated once on the untouched test set.

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

extra_test_results = []

for target in TARGETS:
    features = input_cols(target)

    # Training rows where target and required inputs are available
    train_mask = train[target].notna() & train[features].notna().all(axis=1)
    X_train = train.loc[train_mask, features]
    y_train = train.loc[train_mask, target]

    # Test rows where target and required inputs are available
    test_mask = test[target].notna() & test[features].notna().all(axis=1)
    X_test = test.loc[test_mask, features]
    y_test = test.loc[test_mask, target]

    model = extra_trees_best[target]

    # Refit selected model on all available training data
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5

    extra_test_results.append({
        "model": "Extra-Trees",
        "target": target,
        "n_test": len(y_test),
        "MAE_test": mae,
        "RMSE_test": rmse
    })

extra_test_results = pd.DataFrame(extra_test_results)

extra_test_results

,model,target,n_test,MAE_test,RMSE_test
0,Extra-Trees,yield_strength_MPa,147,28.224427,42.648203
1,Extra-Trees,uts_MPa,142,20.370957,30.856943
2,Extra-Trees,elongation_pct,129,2.041377,3.120220
3,Extra-Trees,charpy_toughness_J,155,14.897561,25.109724


### 6.2 Gradient Boosting Test Evaluation

The tuned Gradient Boosting models are retrained on all available training observations using the hyperparameters selected during cross-validation, then evaluated on the untouched test set.

In [16]:
from sklearn.ensemble import GradientBoostingRegressor

gb_params = {
    "yield_strength_MPa": {
        "learning_rate": 0.15,
        "max_depth": 3,
        "n_estimators": 400
    },
    "uts_MPa": {
        "learning_rate": 0.10,
        "max_depth": 4,
        "n_estimators": 400
    },
    "elongation_pct": {
        "learning_rate": 0.15,
        "max_depth": 3,
        "n_estimators": 300
    },
    "charpy_toughness_J": {
        "learning_rate": 0.15,
        "max_depth": 4,
        "n_estimators": 400
    }
}

In [17]:
gb_test_results = []

for target in TARGETS:
    features = input_cols(target)

    train_mask = train[target].notna() & train[features].notna().all(axis=1)
    X_train = train.loc[train_mask, features]
    y_train = train.loc[train_mask, target]

    test_mask = test[target].notna() & test[features].notna().all(axis=1)
    X_test = test.loc[test_mask, features]
    y_test = test.loc[test_mask, target]

    gb = GradientBoostingRegressor(
        random_state=42,
        **gb_params[target]
    )

    model = build_pipeline(gb)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    gb_test_results.append({
        "model": "Gradient Boosting Tuned",
        "target": target,
        "n_test": len(y_test),
        "MAE_test": mean_absolute_error(y_test, y_pred),
        "RMSE_test": mean_squared_error(y_test, y_pred) ** 0.5
    })

gb_test_results = pd.DataFrame(gb_test_results)

gb_test_results

,model,target,n_test,MAE_test,RMSE_test
0,Gradient Boosting Tuned,yield_strength_MPa,147,26.520748,37.981682
1,Gradient Boosting Tuned,uts_MPa,142,21.293737,31.063316
2,Gradient Boosting Tuned,elongation_pct,129,2.099846,3.192661
3,Gradient Boosting Tuned,charpy_toughness_J,155,16.900254,25.315436


### 6.3 Regression Test Comparison

The final test-set results confirm the cross-validation conclusions. Extra-Trees achieves the lowest MAE for UTS, elongation, and Charpy toughness, while Gradient Boosting performs best for yield strength.

Overall, Extra-Trees remains the strongest regression approach across the four mechanical-property targets.

### 6.4 Final Weld Conformity on the Test Set

The regression predictions are converted into conformity decisions using the same alloy-specific thresholds as in cross-validation. The final weld label is non-conforming if at least one required criterion fails.

In [18]:
def regression_test_predictions(models, train, test):
    preds = {}

    for target in TARGETS:
        features = input_cols(target)

        train_mask = train[target].notna() & train[features].notna().all(axis=1)
        X_train = train.loc[train_mask, features]
        y_train = train.loc[train_mask, target]

        # Predict all test rows with available inputs
        test_mask = test[features].notna().all(axis=1)
        X_test = test.loc[test_mask, features]

        model = models[target]
        model.fit(X_train, y_train)

        pred = pd.DataFrame(index=X_test.index)
        pred["pred"] = model.predict(X_test)

        # For Charpy, conformity must be evaluated at the
        # reference temperature of each alloy family
        if target == "charpy_toughness_J":
            X_ref = X_test.copy()
            X_ref["charpy_temp_C"] = (
                test.loc[X_test.index, "family"]
                .map(THRESHOLDS["charpy_temp_ref_C"])
                .astype(float)
            )
            pred["pred_ref"] = model.predict(X_ref)
        else:
            pred["pred_ref"] = pred["pred"]

        preds[target] = pred

    return preds

In [19]:
true_test_criteria = true_criteria(test)

extra_preds_test = regression_test_predictions(
    extra_trees_best,
    train,
    test
)

gb_models_final = {
    target: build_pipeline(
        GradientBoostingRegressor(
            random_state=42,
            **gb_params[target]
        )
    )
    for target in TARGETS
}

gb_preds_test = regression_test_predictions(
    gb_models_final,
    train,
    test
)

extra_pred_criteria = predicted_criteria(test, extra_preds_test)
gb_pred_criteria = predicted_criteria(test, gb_preds_test)

extra_test_labels = deposit_labels(
    test,
    true_test_criteria,
    extra_pred_criteria
)

gb_test_labels = deposit_labels(
    test,
    true_test_criteria,
    gb_pred_criteria
)

print("Evaluable test deposits:", len(extra_test_labels))
print("Actual non-conforming:", extra_test_labels["true_nc"].sum())

Evaluable test deposits: 38
Actual non-conforming: 12


In [20]:
from sklearn.metrics import (
    f1_score,
    recall_score,
    accuracy_score,
    precision_score
)

def test_label_metrics(name, labels):
    y_true = labels["true_nc"]
    y_pred = labels["pred_nc"]

    return {
        "model": name,
        "n_test": len(labels),
        "n_nc": int(y_true.sum()),
        "F1_nc": f1_score(y_true, y_pred, zero_division=0),
        "recall_nc": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),

        # good = 0 in true_nc/pred_nc
        "precision_good": precision_score(
            1 - y_true,
            1 - y_pred,
            zero_division=0
        )
    }


regression_label_test = pd.DataFrame([
    test_label_metrics("Extra-Trees", extra_test_labels),
    test_label_metrics("Gradient Boosting Tuned", gb_test_labels)
])

regression_label_test

,model,n_test,n_nc,F1_nc,recall_nc,accuracy,precision_good
0,Extra-Trees,38,12,0.727273,0.666667,0.842105,0.857143
1,Gradient Boosting Tuned,38,12,0.761905,0.666667,0.868421,0.862069


### 6.5 Direct SVM Classifier Test Evaluation

The direct SVM classifier is retrained on all labelled training deposits using the hyperparameters selected by cross-validation. It is then evaluated on the same labelled deposits of the untouched test set.

In [21]:
from sklearn.svm import SVC

# deposit_data expects a fold column.
# The fold value is irrelevant for the final test set because no CV is done here.
test_clf = test.copy()
test_clf["fold"] = 0

train_deposits = deposit_data(train)
test_deposits = deposit_data(test_clf)

print("Training deposits:", len(train_deposits))
print("Test deposits:", len(test_deposits))
print("Test non-conforming:", int(test_deposits["true_nc"].sum()))

Training deposits: 175
Test deposits: 38
Test non-conforming: 12


In [22]:
svm_final = build_pipeline(
    SVC(
        kernel="poly",
        C=100.0,
        degree=2,
        coef0=1
    )
)

svm_final.fit(
    train_deposits[DEPOSIT_COLS],
    train_deposits["true_nc"]
)

svm_pred_test = svm_final.predict(
    test_deposits[DEPOSIT_COLS]
)

In [23]:
svm_test_result = test_label_metrics(
    "SVM classifier",
    pd.DataFrame({
        "true_nc": test_deposits["true_nc"],
        "pred_nc": svm_pred_test
    }, index=test_deposits.index)
)

svm_test_result

{'model': 'SVM classifier',
 'n_test': 38,
 'n_nc': 12,
 'F1_nc': 0.5185185185185185,
 'recall_nc': 0.5833333333333334,
 'accuracy': 0.6578947368421053,
 'precision_good': 0.782608695652174}

### 6.6 Final Test-Set Comparison

The direct SVM classifier performs substantially worse on the untouched test set than suggested by its cross-validation results.

Among the three final candidates, Gradient Boosting achieves the strongest overall conformity-classification performance, with the highest F1-score, accuracy, and precision for predicted good welds. Extra-Trees remains close and achieves the same recall for non-conforming welds.

The direct SVM classifier shows a larger drop between cross-validation and test performance, indicating weaker generalization on the held-out test set.

In [24]:
final_label_test = pd.DataFrame([
    regression_label_test.iloc[0].to_dict(),
    regression_label_test.iloc[1].to_dict(),
    svm_test_result
])

final_label_test = final_label_test.sort_values(
    "F1_nc",
    ascending=False
)

final_label_test

,model,n_test,n_nc,F1_nc,recall_nc,accuracy,precision_good
1,Gradient Boosting Tuned,38,12,0.761905,0.666667,0.868421,0.862069
0,Extra-Trees,38,12,0.727273,0.666667,0.842105,0.857143
2,SVM classifier,38,12,0.518519,0.583333,0.657895,0.782609


## 7. Cross-Validation vs Test-Set Performance

The final candidates are compared between cross-validation and the untouched test set. This comparison helps assess whether the models generalize consistently or whether their cross-validation performance was overly optimistic.

In [25]:
# CV results for the three final candidates
cv_final = classification_comparison[
    classification_comparison["model"].isin([
        "Gradient Boosting Tuned",
        "Extra-Trees",
        "SVM classifier"
    ])
][
    [
        "model",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good"
    ]
].copy()

cv_final = cv_final.rename(columns={
    "F1_nc": "F1_CV",
    "recall_nc": "recall_CV",
    "accuracy": "accuracy_CV",
    "precision_good": "precision_good_CV"
})


# Test results
test_final = final_label_test[
    [
        "model",
        "F1_nc",
        "recall_nc",
        "accuracy",
        "precision_good"
    ]
].copy()

test_final = test_final.rename(columns={
    "F1_nc": "F1_test",
    "recall_nc": "recall_test",
    "accuracy": "accuracy_test",
    "precision_good": "precision_good_test"
})


cv_vs_test = cv_final.merge(
    test_final,
    on="model"
)

cv_vs_test

,model,F1_CV,recall_CV,accuracy_CV,precision_good_CV,F1_test,recall_test,accuracy_test,precision_good_test
0,Gradient Boosting Tuned,0.8590,0.8072,0.8743,0.8431,0.761905,0.666667,0.868421,0.862069
1,SVM classifier,0.8415,0.8313,0.8514,0.8511,0.518519,0.583333,0.657895,0.782609
2,Extra-Trees,0.8392,0.7229,0.8686,0.8000,0.727273,0.666667,0.842105,0.857143


### 7.1 Generalization Analysis

Gradient Boosting and Extra-Trees maintain relatively strong performance on the untouched test set, although their F1-scores decrease compared with cross-validation.

Gradient Boosting decreases from an F1-score of 0.859 in cross-validation to 0.762 on the test set, while its accuracy remains very stable (0.874 vs. 0.868). Extra-Trees decreases from 0.839 to 0.727 in F1-score and from 0.869 to 0.842 in accuracy.

In contrast, the direct SVM classifier shows a much larger deterioration. Its F1-score decreases from 0.842 to 0.519 and its accuracy from 0.851 to 0.658. This suggests that the strong cross-validation performance of the direct SVM does not generalize as well to unseen deposits.

These results therefore favor the regression-based approaches, particularly Gradient Boosting and Extra-Trees.

The test classification results must nevertheless be interpreted cautiously because only 38 test deposits have sufficient information to determine all four conformity criteria, of which 12 are non-conforming.

## 8. Final Recommendation

The results suggest using a regression-based approach rather than directly predicting the final conformity class.

For prediction of the individual mechanical properties, Extra-Trees is the strongest overall model. On the untouched test set, it achieves the lowest MAE for UTS (20.37 MPa), elongation (2.04 percentage points), and Charpy toughness (14.90 J). Gradient Boosting performs best for yield strength, with an MAE of 26.52 MPa.

For the final conforming/non-conforming decision, Gradient Boosting provides the best test-set performance among the selected candidates, with an F1-score of 0.762, recall of 0.667, accuracy of 0.868, and precision of 0.862 for welds predicted as conforming.

Therefore, Gradient Boosting is retained as the preferred model for the final weld-quality decision, while Extra-Trees is retained as the strongest model when accurate prediction of the individual mechanical properties is the primary objective.

The direct SVM classifier is not retained because its performance decreases substantially on the untouched test set.

## 9. Limitations

Several limitations should be considered when interpreting the results.

First, the final conformity evaluation is based on only 38 test deposits, including 12 non-conforming deposits. Therefore, classification metrics such as F1-score and recall may vary substantially with only a few additional errors.

Second, the dataset is strongly dominated by C-Mn steels. The CrMo2 and CrMo91 families contain far fewer observations, so conclusions for these alloy families are less reliable than for C-Mn steels.

Third, conformity labels can only be determined for deposits where all four required mechanical criteria are available. This considerably reduces the number of observations available for the final classification task.

Finally, the semi-supervised approaches tested in this project did not provide a clear improvement over supervised learning. The strong difference between labelled and unlabelled data distributions also limits the benefit of using the unlabelled observations directly.

## 10. Conclusion

The study shows that ensemble tree-based models provide the strongest performance for weld-quality prediction.

Extra-Trees gives the best overall prediction of the individual mechanical properties, while Gradient Boosting gives the strongest final conformity classification on the untouched test set.

The final recommended approach is therefore to predict the mechanical properties using supervised regression models and derive the conformity decision from the corresponding alloy-specific engineering thresholds, rather than relying on a direct classifier.